#### 1. Setup

Imports, plot style, and pandas display options.


In [14]:
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.max_columns", 50)

#### 2. Config & Constants

Paths, station list, coordinates, forecast horizons, lookback window.


In [15]:
BASE_DIR = Path("AQI_DATA_024_025")
YEAR_DIRS = ["024", "025"]
STATIONS = ["Bhaktapur", "Khumaltar", "Ratnapark", "Shankhapark"]
PM25_COL = "PM2.5"

COORDS = {
    "Bhaktapur": (27.673762, 85.417528),
    "Khumaltar": (27.645986, 85.323808),
    "Ratnapark": (27.700000, 85.310000),
    "Shankhapark": (27.722654, 85.222836),
}

FORECAST_HORIZONS = [24, 48, 72]
LOOKBACK = 168

OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

EPA_UNHEALTHY = 55.5


#### 3. Helper Functions

`gap_lengths()` and `iqr_outlier_pct()`, reused across later sections.


In [16]:
def gap_lengths(is_missing):
    """Length of every run of consecutive missing values in a boolean series."""
    grp = (~is_missing).cumsum()
    runs = is_missing.groupby(grp).sum()
    return runs[runs > 0].values

def iqr_outlier_pct(s):
    """% of values in s falling outside the 1.5*IQR fence."""
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    return 100 * ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()


#### 4. Load Raw Data

Reads each station/year CSV and combines into one raw table.


In [17]:
def load_file(year_dir, station):
    df = pd.read_csv(BASE_DIR / year_dir / f"{station}.csv", na_values=["NA"])
    df["timestamp"] = pd.to_datetime(df["Date"], format="mixed")
    df["station"] = station
    return df[["timestamp", PM25_COL, "station"]]

raw = pd.concat(
    [load_file(year, station) for year in YEAR_DIRS for station in STATIONS],
    ignore_index=True,
)


#### 5. Raw Dataset Overview

Shape, dtypes, and a peek at the raw rows before reindexing.


In [18]:
print("Raw dataset shape:", raw.shape)
print()
print(raw.dtypes)
raw.head()


Raw dataset shape: (68430, 3)

timestamp    datetime64[ns]
PM2.5               float64
station              object
dtype: object


,timestamp,PM2.5,station
0,2024-01-01 00:00:00,NaN,Bhaktapur
1,2024-01-01 01:00:00,NaN,Bhaktapur
2,2024-01-01 02:00:00,NaN,Bhaktapur
3,2024-01-01 03:00:00,NaN,Bhaktapur
4,2024-01-01 04:00:00,NaN,Bhaktapur


#### 6. Date Range Check

Start and end timestamp of the raw data.


In [19]:
print("Start date:", raw["timestamp"].min())
print("End date:", raw["timestamp"].max())

Start date: 2024-01-01 00:00:00
End date: 2025-12-31 23:00:00


#### 7. Timestamp / Duplicate Check

Invalid timestamps and duplicate (station, timestamp) rows.


In [20]:
print("Invalid timestamps:", raw["timestamp"].isna().sum())
print("Duplicate (station, timestamp) records:", raw.duplicated(["station", "timestamp"]).sum())

Invalid timestamps: 0
Duplicate (station, timestamp) records: 0


#### 8. Record Counts

Raw row count per station, before reindexing.


In [21]:
print("Raw record count per station (pre-reindex):")
print(raw.groupby("station").size())
print()
print("Total raw records (pre-reindex):", len(raw))


Raw record count per station (pre-reindex):
station
Bhaktapur      17544
Khumaltar      17544
Ratnapark      16144
Shankhapark    17198
dtype: int64

Total raw records (pre-reindex): 68430


#### 9. Reindex to Hourly Grid

Fills missing hours as NaN rows, merges station coordinates, saves combined CSV.


In [22]:
full_range = pd.date_range(raw["timestamp"].min(), raw["timestamp"].max(), freq="h")

reindexed = []
for station in STATIONS:
    s = raw.loc[raw.station == station].set_index("timestamp")[PM25_COL]
    s = s[~s.index.duplicated()].reindex(full_range)
    reindexed.append(pd.DataFrame({"timestamp": full_range, PM25_COL: s.values, "station": station}))
raw_full_grid = pd.concat(reindexed, ignore_index=True)

coord_df = pd.DataFrame(COORDS, index=["lat", "lon"]).T.reset_index(names="station")

aqi_data = (
    raw_full_grid.merge(coord_df, on="station")
       .sort_values(["station", "timestamp"])
       .reset_index(drop=True)
)

aqi_data.to_csv(OUT_DIR / "raw_hourly_combined.csv", index=False)

print(aqi_data.shape)
aqi_data.head()

(70176, 5)


,timestamp,PM2.5,station,lat,lon
0,2024-01-01 00:00:00,NaN,Bhaktapur,27.673762,85.417528
1,2024-01-01 01:00:00,NaN,Bhaktapur,27.673762,85.417528
2,2024-01-01 02:00:00,NaN,Bhaktapur,27.673762,85.417528
3,2024-01-01 03:00:00,NaN,Bhaktapur,27.673762,85.417528
4,2024-01-01 04:00:00,NaN,Bhaktapur,27.673762,85.417528


#### 10. Before/After Reindex

Compares record counts pre- and post-reindexing.


In [23]:
print("Before reindexing:")
print(f"  Total records: {len(raw)}")
print()
print("After reindexing:")
print(aqi_data.groupby('station').size().rename('records'))
print(f"  Total records: {len(aqi_data)}")


Before reindexing:
  Total records: 68430

After reindexing:
station
Bhaktapur      17544
Khumaltar      17544
Ratnapark      17544
Shankhapark    17544
Name: records, dtype: int64
  Total records: 70176


#### 11. Timeline Range per Station

Min/max timestamp for each station.


In [24]:
aqi_data.groupby("station")["timestamp"].agg(["min", "max"])


,min,max
station,,
Bhaktapur,2024-01-01,2025-12-31 23:00:00
Khumaltar,2024-01-01,2025-12-31 23:00:00
Ratnapark,2024-01-01,2025-12-31 23:00:00
Shankhapark,2024-01-01,2025-12-31 23:00:00


#### 12. Missing Data Summary

Overall missing count and percentage after reindexing.


In [25]:
print(f"Total records: {len(aqi_data)}")
print(f"Total stations: {aqi_data['station'].nunique()}")

missing_count = aqi_data[PM25_COL].isna().sum()
missing_percentage = aqi_data[PM25_COL].isna().mean() * 100

print(f"Total missing values: {missing_count}")
print(f"Overall missing percentage: {missing_percentage:.2f}%")

Total records: 70176
Total stations: 4
Total missing values: 9182
Overall missing percentage: 13.08%


#### 13. Per-Station Missing Summary

Missing % plus gap episode count and longest gap per station.


In [26]:
station_summary = (
    aqi_data.groupby("station")
    .agg(
        Total_Records=(PM25_COL, "size"),
        Missing_Values=(PM25_COL, lambda x: x.isna().sum()),
    )
)

station_summary["Missing_Percentage"] = (
    station_summary["Missing_Values"] / station_summary["Total_Records"] * 100
).round(2)

def gap_episode_stats(s):
    gl = gap_lengths(s.isna())
    return pd.Series({
        "Gap_Episodes": len(gl),
        "Longest_Gap_h": int(gl.max()) if len(gl) else 0,
    })

gap_stats = aqi_data.groupby("station")[PM25_COL].apply(gap_episode_stats).unstack()
station_summary = station_summary.join(gap_stats)

station_summary

,Total_Records,Missing_Values,Missing_Percentage,Gap_Episodes,Longest_Gap_h
station,,,,,
Bhaktapur,17544,3175,18.10,174,836
Khumaltar,17544,507,2.89,45,54
Ratnapark,17544,3533,20.14,48,1488
Shankhapark,17544,1967,11.21,29,1011
